# chem milestone 2 (food/waste): pre-merge test (2026-10-23)

Branch `chem`. Food/waste as a special case of the general chemistry (`Docs/plan-chem.md` §7). Each check enters through a public door and prints PASS/FAIL:

1. **Parity.** `N_chem=1` matches the milestone-1 golden bit for bit, and an inert `N_chem=2` equals `N_chem=1` bit for bit.
2. **Mechanism.** One-step fixtures: metabolism plus excretion, the sink, toxic death returning food and waste, and the 50/50 split.
3. **Conservation.** The waste mass identity and the energy identity hold every step on an evolving run.
4. **Doors.** `init` and misuse, `run_sim` echo, recipe round trip, gallery override, colormode list.
5. **The full pytest suite.**

In [1]:
import sys, os, json, tempfile, subprocess, warnings, hashlib
from pathlib import Path
import numpy as np
p = Path.cwd().resolve()
while not (p / 'C' / 'evoca.h').exists(): p = p.parent
ROOT = p; sys.path.insert(0, str(ROOT / 'python')); sys.path.insert(0, str(ROOT / 'tests'))
from evoca_py import EvoCA, import_run
def check(name, ok): print(('PASS' if ok else 'FAIL') + ': ' + name); assert ok
def fp(s):
    h = hashlib.sha1()
    for a in (s.get_F(), s.get_f(), s.get_v(), s.get_alive(), s.get_lut_hashes()): h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()
def evolving(N=48, seed=1, n_after=None, **kw):
    rng = np.random.default_rng(seed); pp = dict(food_inc=0.015, m_scale=3.0, gdiff=0.08, tax=0.02, mu_lut=0.01, mu_egene=0.03); pp.update(kw)
    s = EvoCA(); s.init(N, **pp)
    if n_after: s.set_n_chem(n_after)
    s.set_seed(seed); s.state(lut='gol', egenome='uniform', egenome_value=3)
    s.set_v(rng.integers(0, 2, (N, N), dtype=np.uint8)); s.set_f_all(0.5); s.set_F_all(1.0); s.set_alive_halfplane(0)
    return s
W = dict(n_chem=2, wdiff=0.3, waste_sink=0.01, waste_disp=0.2)

## 1. Parity

In [2]:
import chem_parity as cp
ok, _ = cp.compare(cp.load_golden(), cp.run_and_capture(EvoCA()))
check('N_chem=1 == milestone-1 golden (bit for bit)', ok)
f = {}
for n in (None, 2):
    s = evolving(n_after=n); [s.step() for _ in range(300)]; f[n] = fp(s); s.free()
check('inert N_chem=2 == N_chem=1 over 300 steps (bit for bit)', f[None] == f[2])

PASS: N_chem=1 == milestone-1 golden (bit for bit)
PASS: inert N_chem=2 == N_chem=1 over 300 steps (bit for bit)


## 2. Mechanism (constructed one-step fixtures)

In [3]:
def still(N=16, **kw):
    s = EvoCA(); s.init(N, food_inc=0.0, m_scale=0.0, gdiff=0.0, n_chem=2, **kw)
    s.state(lut='gol', egenome='uniform', egenome_value=3); s.set_F_all(0.0); s.set_alive_all(); s.set_f_all(0.5); return s
s = still(tax=0.03, waste_disp=0.25); s.step()
check('metabolism + excretion: f=0.47, w=0.0225, W=0.0075', np.allclose(s.get_f(), 0.47) and np.allclose(s.get_w(), 0.0225, atol=1e-7) and np.allclose(s.get_W(), 0.0075, atol=1e-7)); s.free()
s = still(tax=0.0, waste_sink=0.1); env = np.ctypeslib.as_array(s._lib.evoca_get_chem_env(1), shape=(256,)); env[:] = np.linspace(0, 0.3, 256, dtype=np.float32); b = env.copy(); s.step()
check('sink: W -> max(W - waste_sink, 0)', np.allclose(s.get_W().ravel(), np.maximum(b - 0.1, 0), atol=1e-7)); s.free()
s = still(tax=0.02); w0 = np.full((16, 16), 0.5, np.float32); w0[3, 4] = 0.99; s.set_w(w0); s.step()
check('toxic death at w>=1 returns food (0.48) to F and waste (1.01) to W', s.get_alive().reshape(16, 16)[3, 4] == 0 and abs(s.get_F()[3, 4] - 0.48) < 1e-6 and abs(s.get_W()[3, 4] - 1.01) < 1e-6 and s.get_tox_deaths_last() == 1); s.free()
s = still(N=8, tax=0.0); a = np.zeros((8, 8), np.uint8); a[4, 4] = 1; s.set_alive(a); s.set_f_all(1.0); s.set_w(0.4); s.step(); al = s.get_alive().reshape(8, 8) == 1
check('reproduction splits food and waste 50/50', al.sum() == 2 and np.allclose(s.get_w()[al], 0.2) and np.allclose(s.get_f()[al], 0.5)); s.free()

PASS: metabolism + excretion: f=0.47, w=0.0225, W=0.0075
PASS: sink: W -> max(W - waste_sink, 0)
PASS: toxic death at w>=1 returns food (0.48) to F and waste (1.01) to W
PASS: reproduction splits food and waste 50/50


## 3. Conservation (evolving run, N=48, 300 steps)

In [4]:
s = evolving(**W)
mass = lambda: s.get_W().astype(np.float64).sum() + s.get_w().astype(np.float64).sum()
m0, fl0 = mass(), s.get_chem_flux(1); d0 = s.get_dissipation(); e0 = d0['storage_env'] + d0['storage_priv']
wr, er, tox = [], [], 0
for _ in range(300):
    s.step(); tox += s.get_tox_deaths_last(); m, fl = mass(), s.get_chem_flux(1); d = s.get_dissipation(); e = d['storage_env'] + d['storage_priv']
    wr.append((m - m0) - ((fl[0] - fl0[0]) - (fl[1] - fl0[1]))); er.append(d['source_step'] - (e - e0 + d['diss_step'])); m0, fl0, e0 = m, fl, e
alive = s.get_alive().mean(); s.free()
check(f'waste mass identity every step (max |residual| {max(map(abs, wr)):.1e})', max(map(abs, wr)) < 2e-3)
check(f'energy identity every step (max |residual| {max(map(abs, er)):.1e})', max(map(abs, er)) < 2e-3)
check(f'non-vacuous: alive {alive:.2f}, toxic deaths {tox}', alive > 0.05 and tox > 0)

PASS: waste mass identity every step (max |residual| 5.5e-05)
PASS: energy identity every step (max |residual| 3.1e-05)
PASS: non-vacuous: alive 0.85, toxic deaths 2082


## 4. Doors

In [5]:
s = EvoCA(); s.init(16, **W)
check('init(n_chem=2,...) applies the preset', s.get_metab_product() == 1 and s.get_chem_energy(1) == 0.0 and np.allclose(s.get_chem_params(1), (0, 0.01, 0.3, 0.2, 1.0))); s.free()
try: EvoCA().init(16, wdiff=0.1); bad = False
except ValueError: bad = True
check('waste params without n_chem=2 are refused', bad)
from evoca_explore import run_sim
with warnings.catch_warnings():
    warnings.simplefilter('error'); r = run_sim(dict(food_inc=0.015, m_scale=3.0, tax=0.02, **W), n_steps=50, sample_every=25, N=16, seed=1, shadow=False)
check('run_sim forwards and echoes the waste params', (r['param_n_chem'], r['param_wdiff'], r['param_waste_sink'], r['param_waste_disp']) == (2, 0.3, 0.01, 0.2))
with tempfile.TemporaryDirectory() as t:
    s = EvoCA(); s.init(16, m_scale=3.0, **W); s.state(lut='gol', egenome='uniform'); path = s.export_recipe('_t', runs_dir=t); s.free()
    sim, _ = import_run(path); okr = sim.get_n_chem() == 2 and np.allclose(sim.get_chem_params(1)[1:4], (0.01, 0.3, 0.2)); sim.free()
    from evoca_gallery import Entry
    sim, _ = Entry(recipe=path, overrides={'waste_sink': 0.03}).load(N=16); okg = abs(sim.get_chem_params(1)[1] - 0.03) < 1e-7; sim.free()
check('recipe round trip', okr); check('gallery override reaches C', okg)
from colormodes import COLOR_MODES
s = EvoCA(); s.init(8); check('colormode list matches C (7, food-waste last)', len(COLOR_MODES) == s._lib.evoca_colormode_count() == 7 and COLOR_MODES[6] == 'food-waste'); s.free()

PASS: init(n_chem=2,...) applies the preset
PASS: waste params without n_chem=2 are refused


PASS: run_sim forwards and echoes the waste params


PASS: recipe round trip
PASS: gallery override reaches C
PASS: colormode list matches C (7, food-waste last)


## 5. Full suite

In [6]:
out = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT / 'tests')], capture_output=True, text=True, cwd=ROOT).stdout.strip().splitlines()[-1]
print(out); check('pytest suite passes', 'failed' not in out and 'passed' in out)
print('ALL PASS')

184 passed in 5.14s
PASS: pytest suite passes
ALL PASS
